# Tests de mini-funcionalidades de OP-10: Get trips from flows

Este notebook se usa para probar helpers y bloques internos de `get_trips_from_flows()` antes de hacer tests más integrados de la función completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`
- las pruebas de este notebook no reemplazan tests integrados

## Bloque 0. Preparación

### Imports generales

In [1]:
from copy import deepcopy

import pandas as pd
import h3

### Imports del módulo

In [2]:
from pylondrina.datasets import FlowDataset, TripDataset
from pylondrina.errors import PylondrinaError
from pylondrina.reports import Issue
from pylondrina.schema import TripSchema
from pylondrina.queries.flows import (
    _resolve_correspondence_source,
    _extract_correspondence_from_flow_to_trips,
    _reconstruct_correspondence_from_trips,
    _finalize_flow_trip_correspondence,
    _resolve_reconstruction_join_info,
    _apply_h3_rollup_if_needed,
    _truncate_query_issues,
    _coerce_datetime_series,
    _make_window_start,
    _make_window_end,
    _safe_sort_correspondence_df,
    _unique_non_null_values,
)

### Helpers de apoyo para test

In [3]:
def h3_from_latlon(lat: float, lon: float, res: int = 8) -> str:
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, res)
    return h3.geo_to_h3(lat, lon, res)


def h3_to_parent(cell: str, res: int) -> str:
    if hasattr(h3, "cell_to_parent"):
        return h3.cell_to_parent(cell, res)
    return h3.h3_to_parent(cell, res)


def assert_issue_codes(issues: list[Issue], expected_codes: list[str]) -> None:
    observed = [issue.code for issue in issues]
    assert observed == expected_codes, f"Esperados={expected_codes}, observados={observed}"


def make_request_ctx(
    *,
    max_issues: int = 1000,
    n_flows_input: int = 0,
    n_trips_input: int | None = None,
    used_source: str | None = None,
    reconstruction_attempted: bool = False,
) -> dict:
    return {
        "max_issues": max_issues,
        "n_flows_input": n_flows_input,
        "n_trips_input": n_trips_input,
        "used_source": used_source,
        "reconstruction_attempted": reconstruction_attempted,
    }


def make_op13_test_schema() -> TripSchema:
    return TripSchema(version="0.1.0", fields={}, required=[])


def make_op13_test_tripdataset(res: int = 8) -> TripDataset:
    # Coordenadas de apoyo solo para generar H3 válidos
    points = {
        "A": (-33.4500, -70.6600),
        "B": (-33.4400, -70.6400),
        "C": (-33.4600, -70.6200),
        "D": (-33.4700, -70.6100),
        "E": (-33.4300, -70.6000),
        "F": (-33.4200, -70.5800),
    }
    cells = {k: h3_from_latlon(lat, lon, res) for k, (lat, lon) in points.items()}

    df = pd.DataFrame(
        [
            {
                "movement_id": "m0",
                "trip_id": "t0",
                "origin_h3_index": cells["A"],
                "destination_h3_index": cells["B"],
                "mode": "bus",
                "purpose": "work",
                "origin_time_utc": "2026-01-01T08:05:00Z",
                "destination_time_utc": "2026-01-01T08:25:00Z",
            },
            {
                "movement_id": "m1",
                "trip_id": "t1",
                "origin_h3_index": cells["A"],
                "destination_h3_index": cells["B"],
                "mode": "bus",
                "purpose": "work",
                "origin_time_utc": "2026-01-01T08:15:00Z",
                "destination_time_utc": "2026-01-01T08:35:00Z",
            },
            {
                "movement_id": "m2",
                "trip_id": "t2",
                "origin_h3_index": cells["A"],
                "destination_h3_index": cells["C"],
                "mode": "metro",
                "purpose": "study",
                "origin_time_utc": "2026-01-01T09:10:00Z",
                "destination_time_utc": "2026-01-01T09:35:00Z",
            },
            {
                "movement_id": "m3",
                "trip_id": "t3",
                "origin_h3_index": cells["D"],
                "destination_h3_index": cells["E"],
                "mode": "bus",
                "purpose": "work",
                "origin_time_utc": "2026-01-01T08:20:00Z",
                "destination_time_utc": "2026-01-01T08:50:00Z",
            },
        ]
    )

    return TripDataset(
        data=df,
        schema=make_op13_test_schema(),
        metadata={
            "dataset_id": "trips_op13_test",
            "is_validated": True,
            "temporal": {"tier": "tier_1"},
            "events": [],
        },
    )


def make_op13_test_flowdataset(
    *,
    include_flow_to_trips: bool = True,
    duplicate_direct_pairs: bool = False,
    include_extra_unmatched_flow: bool = True,
    source_trips: TripDataset | None = None,
    flow_resolution: int = 8,
) -> FlowDataset:
    trips_for_build = source_trips if source_trips is not None else make_op13_test_tripdataset(res=8)
    tdf = trips_for_build.data.copy()

    def maybe_roll(cell: str) -> str:
        return cell if flow_resolution == 8 else h3_to_parent(cell, flow_resolution)

    flows_rows = [
        {
            "flow_id": "f_ab_bus_work_h08",
            "origin_h3_index": maybe_roll(tdf.loc[0, "origin_h3_index"]),
            "destination_h3_index": maybe_roll(tdf.loc[0, "destination_h3_index"]),
            "window_start_utc": pd.Timestamp("2026-01-01 08:00:00"),
            "window_end_utc": pd.Timestamp("2026-01-01 09:00:00"),
            "mode": "bus",
            "purpose": "work",
            "flow_count": 2,
            "flow_value": 2.0,
        },
        {
            "flow_id": "f_ac_metro_study_h09",
            "origin_h3_index": maybe_roll(tdf.loc[2, "origin_h3_index"]),
            "destination_h3_index": maybe_roll(tdf.loc[2, "destination_h3_index"]),
            "window_start_utc": pd.Timestamp("2026-01-01 09:00:00"),
            "window_end_utc": pd.Timestamp("2026-01-01 10:00:00"),
            "mode": "metro",
            "purpose": "study",
            "flow_count": 1,
            "flow_value": 1.0,
        },
    ]

    if include_extra_unmatched_flow:
        extra_origin = h3_from_latlon(-33.4100, -70.5700, 8)
        extra_dest = h3_from_latlon(-33.4000, -70.5500, 8)
        flows_rows.append(
            {
                "flow_id": "f_extra_walk_leisure_h10",
                "origin_h3_index": maybe_roll(extra_origin),
                "destination_h3_index": maybe_roll(extra_dest),
                "window_start_utc": pd.Timestamp("2026-01-01 10:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-01 11:00:00"),
                "mode": "walk",
                "purpose": "leisure",
                "flow_count": 5,
                "flow_value": 5.0,
            }
        )

    flows_df = pd.DataFrame(flows_rows)

    flow_to_trips = None
    if include_flow_to_trips:
        flow_to_trips = pd.DataFrame(
            [
                {"flow_id": "f_ab_bus_work_h08", "movement_id": "m0"},
                {"flow_id": "f_ab_bus_work_h08", "movement_id": "m1"},
                {"flow_id": "f_ac_metro_study_h09", "movement_id": "m2"},
            ]
        )
        if duplicate_direct_pairs:
            flow_to_trips = pd.concat(
                [flow_to_trips, flow_to_trips.iloc[[0]]],
                ignore_index=True,
            )

    aggregation_spec = {
        "h3_resolution": flow_resolution,
        "group_by": ["mode", "purpose"],
        "time_aggregation": "hour",
        "time_basis": "origin",
        "effective_flow_keys": [
            "origin_h3_index",
            "destination_h3_index",
            "window_start_utc",
            "window_end_utc",
            "mode",
            "purpose",
        ],
    }

    return FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips,
        aggregation_spec=aggregation_spec,
        source_trips=source_trips,
        metadata={
            "dataset_id": "flows_op13_test",
            "events": [],
        },
        provenance={
            "derived_from": [
                {"type": "trips", "dataset_id": trips_for_build.metadata.get("dataset_id")}
            ]
        },
    )

### Configuración visual

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")

Imports OK


## Bloque 1. Utilidades internas de uso general

Aquí se prueban primero las piezas internas más sensibles que sostienen la reconstrucción: resolución de llaves efectivas, roll-up H3, ventanas temporales, truncamiento, orden estable y unicidad observable. Esto deja una base sólida antes de entrar a los helpers principales.

### Test 1.1 - _resolve_reconstruction_join_info caso feliz

Qué prueba: deriva correctamente las llaves efectivas desde aggregation_spec, incluyendo group_by y ventanas temporales.


In [5]:
flows = make_op13_test_flowdataset(include_flow_to_trips=False, include_extra_unmatched_flow=False)
issues = []
ctx = make_request_ctx(
    n_flows_input=len(flows.flows),
    used_source="trips_argument",
    reconstruction_attempted=True,
    n_trips_input=4,
)

join_info = _resolve_reconstruction_join_info(
    flows.flows,
    aggregation_spec=flows.aggregation_spec,
    issues=issues,
    request_ctx=ctx,
)

assert issues == []
assert join_info["join_key_columns"] == [
    "origin_h3_index",
    "destination_h3_index",
    "window_start_utc",
    "window_end_utc",
    "mode",
    "purpose",
]
assert join_info["group_by"] == ["mode", "purpose"]
assert join_info["window_columns"] == ["window_start_utc", "window_end_utc"]
assert join_info["time_aggregation"] == "hour"
assert join_info["time_basis"] == "origin"
assert join_info["h3_resolution_target"] == 8

### Test 1.2 - _resolve_reconstruction_join_info fatal por group_by no interpretable

Qué prueba: aborta cuando aggregation_spec no permite reconstruir una llave de agregación confiable.

In [6]:
flows = make_op13_test_flowdataset(include_flow_to_trips=False, include_extra_unmatched_flow=False)
issues = []
ctx = make_request_ctx(
    n_flows_input=len(flows.flows),
    used_source="trips_argument",
    reconstruction_attempted=True,
    n_trips_input=4,
)

bad_agg = dict(flows.aggregation_spec)
bad_agg["group_by"] = "mode"  # debe ser lista/tupla, no scalar

raised = None
try:
    _resolve_reconstruction_join_info(
        flows.flows,
        aggregation_spec=bad_agg,
        issues=issues,
        request_ctx=ctx,
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, PylondrinaError)
assert raised.issue is not None
assert raised.issue.code == "GET_TRIPS_FROM_FLOWS.RECON.AGGREGATION_KEYS_UNRECOVERABLE"
assert raised.issue.details["reason"] == "group_by_not_sequence"

### Test 1.3 - _apply_h3_rollup_if_needed caso feliz

Qué prueba: cuando la resolución objetivo es más gruesa, hace roll-up H3 y no muta el input.

In [7]:
trips = make_op13_test_tripdataset(res=8)
issues = []
ctx = make_request_ctx(
    used_source="trips_argument",
    reconstruction_attempted=True,
    n_trips_input=len(trips.data),
)

before = trips.data.copy(deep=True)

rolled = _apply_h3_rollup_if_needed(
    trips.data.loc[:, ["origin_h3_index", "destination_h3_index"]].copy(),
    target_resolution=7,
    issues=issues,
    request_ctx=ctx,
)

assert issues == []
assert rolled is not None
assert rolled.loc[0, "origin_h3_index"] == h3_to_parent(before.loc[0, "origin_h3_index"], 7)
assert rolled.loc[0, "destination_h3_index"] == h3_to_parent(before.loc[0, "destination_h3_index"], 7)

# El input original no debe mutar
pd.testing.assert_frame_equal(trips.data, before)

### Test 1.4 - _apply_h3_rollup_if_needed fatal por resolución objetivo más fina

Qué prueba: aborta si el flow exige una resolución H3 más fina que la disponible en trips.

In [8]:
trips = make_op13_test_tripdataset(res=8)
issues = []
ctx = make_request_ctx(
    used_source="trips_argument",
    reconstruction_attempted=True,
    n_trips_input=len(trips.data),
)

raised = None
try:
    _apply_h3_rollup_if_needed(
        trips.data.loc[:, ["origin_h3_index", "destination_h3_index"]].copy(),
        target_resolution=9,
        issues=issues,
        request_ctx=ctx,
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, PylondrinaError)
assert raised.issue is not None
assert raised.issue.code == "GET_TRIPS_FROM_FLOWS.RECON.AGGREGATION_KEYS_UNRECOVERABLE"
assert raised.issue.details["reason"] == "target_h3_resolution_finer_than_input"

### Test 1.5 - _coerce_datetime_series, _make_window_start y _make_window_end

Qué prueba: normalización temporal a UTC naive y construcción correcta de ventanas horarias.

In [9]:
series = pd.Series(
    [
        "2026-01-01T08:05:00Z",
        "2026-01-01T08:15:00-03:00",
    ]
)

coerced = _coerce_datetime_series(series)
window_start = _make_window_start(coerced, "hour")
window_end = _make_window_end(window_start, "hour")

assert str(coerced.iloc[0]) == "2026-01-01 08:05:00"
assert str(coerced.iloc[1]) == "2026-01-01 11:15:00"
assert str(window_start.iloc[0]) == "2026-01-01 08:00:00"
assert str(window_start.iloc[1]) == "2026-01-01 11:00:00"
assert str(window_end.iloc[0]) == "2026-01-01 09:00:00"
assert str(window_end.iloc[1]) == "2026-01-01 12:00:00"

### Test 1.6 - _truncate_query_issues

Qué prueba: truncamiento contractual de issues y bloque limits consistente.

In [10]:
issues_all = [
    Issue(level="warning", code="GET_TRIPS_FROM_FLOWS.SOURCE.PREFERRED_SOURCE_UNUSABLE", message="w1"),
    Issue(level="warning", code="GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE", message="w2"),
    Issue(level="warning", code="GET_TRIPS_FROM_FLOWS.OUTPUT.EMPTY_RESULT", message="w3"),
]

retained, limits = _truncate_query_issues(issues_all, max_issues=2)

assert len(retained) == 2
assert retained[-1].code == "GET_TRIPS_FROM_FLOWS.REPORT.ISSUES_TRUNCATED"

assert limits["max_issues"] == 2
assert limits["issues_truncated"] is True
assert limits["n_issues_detected_total"] == 3
assert limits["n_issues_emitted"] == 2

### Test 1.7 - _safe_sort_correspondence_df y _unique_non_null_values

Qué prueba: orden estable contractual y unicidad preservando orden de aparición.

In [11]:
df = pd.DataFrame(
    [
        {"flow_id": "f_b", "movement_id": "m2"},
        {"flow_id": "f_a", "movement_id": "m3"},
        {"flow_id": "f_a", "movement_id": "m1"},
    ]
)

sorted_df = _safe_sort_correspondence_df(df)
assert sorted_df["flow_id"].tolist() == ["f_a", "f_a", "f_b"]
assert sorted_df["movement_id"].tolist() == ["m1", "m3", "m2"]

vals = _unique_non_null_values([None, "m1", "m2", "m1", pd.NA, "m3", "m2"])
assert vals == ["m1", "m2", "m3"]

## Bloque 2. Helpers principales del pipeline

Aquí ya se prueban directamente los cuatro helpers principales que cerraste para OP-13: resolución de fuente, consumo de flow_to_trips, reconstrucción exacta desde trips y normalización/cobertura final.

### Test 2.1 - _resolve_correspondence_source usa flow_to_trips si es usable

Qué prueba: prioridad correcta de la fuente directa cuando el auxiliar ya existe con estructura mínima válida.

In [12]:
flows = make_op13_test_flowdataset(
    include_flow_to_trips=True,
    duplicate_direct_pairs=False,
    include_extra_unmatched_flow=False,
)
issues = []
ctx = make_request_ctx(n_flows_input=len(flows.flows))

used_source, source_obj, reconstruction_attempted, n_trips_input = _resolve_correspondence_source(
    flows,
    trips=None,
    issues=issues,
    request_ctx=ctx,
)

assert used_source == "flow_to_trips"
assert reconstruction_attempted is False
assert n_trips_input is None
pd.testing.assert_frame_equal(source_obj, flows.flow_to_trips)
assert issues == []

### Test 2.2 - _resolve_correspondence_source degrada a trips_argument

Qué prueba: si flow_to_trips existe pero no es usable, cae al argumento trips y deja warning agregado.

In [13]:
trips = make_op13_test_tripdataset()
flows = make_op13_test_flowdataset(
    include_flow_to_trips=True,
    include_extra_unmatched_flow=False,
)
flows.flow_to_trips = flows.flow_to_trips.loc[:, ["flow_id"]].copy()  # rompe contrato mínimo

issues = []
ctx = make_request_ctx(n_flows_input=len(flows.flows))

used_source, source_obj, reconstruction_attempted, n_trips_input = _resolve_correspondence_source(
    flows,
    trips=trips,
    issues=issues,
    request_ctx=ctx,
)

assert used_source == "trips_argument"
assert reconstruction_attempted is True
assert n_trips_input == len(trips.data)
assert source_obj is trips
assert_issue_codes(
    issues,
    ["GET_TRIPS_FROM_FLOWS.SOURCE.PREFERRED_SOURCE_UNUSABLE"],
)

### Test 2.3 - _resolve_correspondence_source usa flows.source_trips como tercer fallback

Qué prueba: respeta la tercera prioridad cuando no hay flow_to_trips usable ni trips explícito.

In [15]:
source_trips = make_op13_test_tripdataset()
flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=False,
    source_trips=source_trips,
)

issues = []
ctx = make_request_ctx(n_flows_input=len(flows.flows))

used_source, source_obj, reconstruction_attempted, n_trips_input = _resolve_correspondence_source(
    flows,
    trips=None,
    issues=issues,
    request_ctx=ctx,
)

assert used_source == "flows.source_trips"
assert reconstruction_attempted is True
assert n_trips_input == len(source_trips.data)
assert source_obj is source_trips
assert issues == []

### Test 2.4 - _resolve_correspondence_source fatal sin ninguna fuente usable

Qué prueba: aborta por precondición cuando no existe de dónde construir la correspondencia.

In [16]:
flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=False,
    source_trips=None,
)

issues = []
ctx = make_request_ctx(n_flows_input=len(flows.flows))

raised = None
try:
    _resolve_correspondence_source(
        flows,
        trips=None,
        issues=issues,
        request_ctx=ctx,
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, PylondrinaError)
assert raised.issue is not None
assert raised.issue.code == "GET_TRIPS_FROM_FLOWS.SOURCE.NO_USABLE_SOURCE"

### Test 2.5 - _extract_correspondence_from_flow_to_trips deduplica pares exactos

Qué prueba: usa solo la estructura mínima, deduplica exactos y deja warning agregado.

In [17]:
flows = make_op13_test_flowdataset(
    include_flow_to_trips=True,
    duplicate_direct_pairs=True,
    include_extra_unmatched_flow=False,
)

issues = []
ctx = make_request_ctx(
    n_flows_input=len(flows.flows),
    used_source="flow_to_trips",
    reconstruction_attempted=False,
)

provisional = _extract_correspondence_from_flow_to_trips(
    flows.flow_to_trips,
    issues=issues,
    request_ctx=ctx,
)

assert provisional.columns.tolist() == ["flow_id", "movement_id"]
assert len(provisional) == 3
assert provisional["movement_id"].tolist() == ["m0", "m1", "m2"]

assert_issue_codes(
    issues,
    ["GET_TRIPS_FROM_FLOWS.SOURCE.DUPLICATE_PAIRS_NORMALIZED"],
)

### Test 2.6 - _reconstruct_correspondence_from_trips caso feliz

Qué prueba: reconstrucción exacta por OD + ventanas temporales + group_by, con trip_id opcional y universo completo de movement_id.

In [18]:
trips = make_op13_test_tripdataset()
flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=True,
)

issues = []
ctx = make_request_ctx(
    n_flows_input=len(flows.flows),
    n_trips_input=len(trips.data),
    used_source="trips_argument",
    reconstruction_attempted=True,
)

trips_before = trips.data.copy(deep=True)

provisional, movement_universe, join_info = _reconstruct_correspondence_from_trips(
    flows.flows,
    trips,
    aggregation_spec=flows.aggregation_spec,
    issues=issues,
    request_ctx=ctx,
)

assert issues == []
assert provisional.columns.tolist() == ["flow_id", "movement_id", "trip_id"]

expected_pairs = [
    ("f_ab_bus_work_h08", "m0", "t0"),
    ("f_ab_bus_work_h08", "m1", "t1"),
    ("f_ac_metro_study_h09", "m2", "t2"),
]
assert list(provisional.itertuples(index=False, name=None)) == expected_pairs

assert movement_universe == ["m0", "m1", "m2", "m3"]
assert join_info["join_key_columns"] == [
    "origin_h3_index",
    "destination_h3_index",
    "window_start_utc",
    "window_end_utc",
    "mode",
    "purpose",
]

# No muta trips.data
pd.testing.assert_frame_equal(trips.data, trips_before)

### Test 2.7 - _reconstruct_correspondence_from_trips fatal por columnas mínimas ausentes

Qué prueba: aborta si trips no tiene las columnas necesarias para reproducir la llave efectiva.

In [19]:
trips = make_op13_test_tripdataset()
trips_bad = TripDataset(
    data=trips.data.drop(columns=["purpose"]).copy(),
    schema=trips.schema,
    metadata=deepcopy(trips.metadata),
)

flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=False,
)

issues = []
ctx = make_request_ctx(
    n_flows_input=len(flows.flows),
    n_trips_input=len(trips_bad.data),
    used_source="trips_argument",
    reconstruction_attempted=True,
)

raised = None
try:
    _reconstruct_correspondence_from_trips(
        flows.flows,
        trips_bad,
        aggregation_spec=flows.aggregation_spec,
        issues=issues,
        request_ctx=ctx,
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, PylondrinaError)
assert raised.issue is not None
assert raised.issue.code == "GET_TRIPS_FROM_FLOWS.RECON.MISSING_REQUIRED_COLUMNS"
assert "purpose" in raised.issue.details["missing_columns"]

### Test 2.8 - _finalize_flow_trip_correspondence caso feliz

Qué prueba: normalización final contractual, deduplicación, orden estable, descarte de flow_id ajenos al dataset y summary sin warnings.

In [20]:
flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=False,
)

provisional = pd.DataFrame(
    [
        {"flow_id": "f_ac_metro_study_h09", "movement_id": "m2", "trip_id": "t2"},
        {"flow_id": "f_ab_bus_work_h08", "movement_id": "m1", "trip_id": "t1"},
        {"flow_id": "f_ab_bus_work_h08", "movement_id": "m0", "trip_id": "t0"},
        {"flow_id": "f_ab_bus_work_h08", "movement_id": "m0", "trip_id": "t0"},   # duplicado exacto
        {"flow_id": "f_unknown", "movement_id": "mx", "trip_id": "tx"},            # flow_id ajeno
    ]
)

issues = []
ctx = make_request_ctx(
    n_flows_input=len(flows.flows),
    used_source="trips_argument",
    reconstruction_attempted=True,
    n_trips_input=3,
)

final_df, summary = _finalize_flow_trip_correspondence(
    provisional,
    flows_df=flows.flows,
    movement_universe=["m2", "m1", "m0"],
    issues=issues,
    request_ctx=ctx,
    join_info={
        "join_key_columns": [
            "origin_h3_index",
            "destination_h3_index",
            "window_start_utc",
            "window_end_utc",
            "mode",
            "purpose",
        ],
        "group_by": ["mode", "purpose"],
        "window_columns": ["window_start_utc", "window_end_utc"],
    },
)

assert issues == []
assert final_df.columns.tolist() == ["flow_id", "movement_id", "trip_id"]
assert list(final_df.itertuples(index=False, name=None)) == [
    ("f_ab_bus_work_h08", "m0", "t0"),
    ("f_ab_bus_work_h08", "m1", "t1"),
    ("f_ac_metro_study_h09", "m2", "t2"),
]

assert summary == {
    "n_rows_out": 3,
    "n_unique_flows_out": 2,
    "n_unique_movements_out": 3,
    "n_unmatched_flows": 0,
    "n_unmatched_movements": 0,
}

### Test 2.9 - _finalize_flow_trip_correspondence cobertura parcial

Qué prueba: emite warning cuando hay flows sin correspondencia y movements del universo que no quedaron asociados.

In [21]:
flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=True,
)

provisional = pd.DataFrame(
    [
        {"flow_id": "f_ab_bus_work_h08", "movement_id": "m0", "trip_id": "t0"},
        {"flow_id": "f_ab_bus_work_h08", "movement_id": "m1", "trip_id": "t1"},
        {"flow_id": "f_ac_metro_study_h09", "movement_id": "m2", "trip_id": "t2"},
    ]
)

issues = []
ctx = make_request_ctx(
    n_flows_input=len(flows.flows),
    used_source="trips_argument",
    reconstruction_attempted=True,
    n_trips_input=4,
)

final_df, summary = _finalize_flow_trip_correspondence(
    provisional,
    flows_df=flows.flows,
    movement_universe=["m0", "m1", "m2", "m3"],
    issues=issues,
    request_ctx=ctx,
    join_info={
        "join_key_columns": [
            "origin_h3_index",
            "destination_h3_index",
            "window_start_utc",
            "window_end_utc",
            "mode",
            "purpose",
        ],
        "group_by": ["mode", "purpose"],
        "window_columns": ["window_start_utc", "window_end_utc"],
    },
)

assert len(final_df) == 3
assert summary["n_unmatched_flows"] == 1
assert summary["n_unmatched_movements"] == 1

assert_issue_codes(
    issues,
    ["GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE"],
)

### Test 2.10 - _finalize_flow_trip_correspondence resultado vacío

Qué prueba: si la operación fue interpretable pero no quedó ninguna correspondencia, retorna vacío y emite warning.

In [22]:
flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=False,
)

provisional = pd.DataFrame(columns=["flow_id", "movement_id"])

issues = []
ctx = make_request_ctx(
    n_flows_input=len(flows.flows),
    used_source="trips_argument",
    reconstruction_attempted=True,
    n_trips_input=2,
)

final_df, summary = _finalize_flow_trip_correspondence(
    provisional,
    flows_df=flows.flows,
    movement_universe=["m0", "m1"],
    issues=issues,
    request_ctx=ctx,
    join_info={
        "join_key_columns": [
            "origin_h3_index",
            "destination_h3_index",
            "window_start_utc",
            "window_end_utc",
            "mode",
            "purpose",
        ],
        "group_by": ["mode", "purpose"],
        "window_columns": ["window_start_utc", "window_end_utc"],
    },
)

assert final_df.empty is True
assert summary == {
    "n_rows_out": 0,
    "n_unique_flows_out": 0,
    "n_unique_movements_out": 0,
    "n_unmatched_flows": 2,
    "n_unmatched_movements": 2,
}

assert_issue_codes(
    issues,
    ["GET_TRIPS_FROM_FLOWS.OUTPUT.EMPTY_RESULT"],
)

## Bloque 3: Smoke tests integrados de get_trips_from_flows

### 3.1 - Fixtures mínimas reutilizables

Qué prepara: utilidades chicas para tomar snapshots del estado de entrada y mantener legibles los smoke tests.
Estos tests asumen que ya existen las fixtures/helpers de los bloques anteriores:

In [23]:
from copy import deepcopy

import pandas as pd

from pylondrina.errors import PylondrinaError
from pylondrina.queries.flows import get_trips_from_flows


def snapshot_flowdataset_state(flows):
    return {
        "flows": flows.flows.copy(deep=True),
        "flow_to_trips": None if flows.flow_to_trips is None else flows.flow_to_trips.copy(deep=True),
        "aggregation_spec": deepcopy(flows.aggregation_spec),
        "metadata": deepcopy(flows.metadata),
        "provenance": deepcopy(flows.provenance),
        "source_trips_ref": flows.source_trips,
    }


def snapshot_tripdataset_state(trips):
    return {
        "data": trips.data.copy(deep=True),
        "schema": trips.schema,
        "schema_effective": getattr(trips, "schema_effective", None),
        "metadata": deepcopy(trips.metadata),
        "provenance": deepcopy(getattr(trips, "provenance", None)),
    }

### Test 3.2 - Happy path mínimo integrado usando flow_to_trips

Qué prueba: el camino feliz completo de la función pública cuando la fuente directa ya existe y es usable. Verifica:

- salida tabular mínima correcta,
- OperationReport correcto,
- parameters y summary,
- ausencia total de side effects,
- no mutación del FlowDataset de entrada.

In [25]:
flows = make_op13_test_flowdataset(
    include_flow_to_trips=True,
    duplicate_direct_pairs=False,
    include_extra_unmatched_flow=False,
)

flows_before = snapshot_flowdataset_state(flows)

correspondence_df, report = get_trips_from_flows(
    flows,
    max_issues=20,
)

# Salida tabular mínima
assert correspondence_df.columns.tolist() == ["flow_id", "movement_id"]
assert list(correspondence_df.itertuples(index=False, name=None)) == [
    ("f_ab_bus_work_h08", "m0"),
    ("f_ab_bus_work_h08", "m1"),
    ("f_ac_metro_study_h09", "m2"),
]

# Reporte
assert report.ok is True
assert report.parameters == {
    "max_issues": 20,
    "used_source": "flow_to_trips",
    "reconstruction_attempted": False,
    "n_flows_input": 2,
    "n_trips_input": None,
}
assert report.summary == {
    "n_rows_out": 3,
    "n_unique_flows_out": 2,
    "n_unique_movements_out": 3,
    "n_unmatched_flows": 0,
    "n_unmatched_movements": 0,
}
assert report.issues == []

# No side effects sobre flows
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
pd.testing.assert_frame_equal(flows.flow_to_trips, flows_before["flow_to_trips"])
assert flows.aggregation_spec == flows_before["aggregation_spec"]
assert flows.metadata == flows_before["metadata"]
assert flows.provenance == flows_before["provenance"]
assert flows.source_trips is flows_before["source_trips_ref"]

display(correspondence_df)
display(report.summary)
display(report.parameters)

print("\nFlujos y viajes obtenidos:")
display(flows.flows)
display(correspondence_df)

,flow_id,movement_id
0,f_ab_bus_work_h08,m0
1,f_ab_bus_work_h08,m1
2,f_ac_metro_study_h09,m2


{'n_rows_out': 3,
 'n_unique_flows_out': 2,
 'n_unique_movements_out': 3,
 'n_unmatched_flows': 0,
 'n_unmatched_movements': 0}

{'max_issues': 20,
 'used_source': 'flow_to_trips',
 'reconstruction_attempted': False,
 'n_flows_input': 2,
 'n_trips_input': None}


Flujos y viajes obtenidos:


,flow_id,origin_h3_index,destination_h3_index,window_start_utc,window_end_utc,mode,purpose,flow_count,flow_value
0,f_ab_bus_work_h08,88b2c55417fffff,88b2c554cdfffff,2026-01-01 08:00:00,2026-01-01 09:00:00,bus,work,2,2.0
1,f_ac_metro_study_h09,88b2c55417fffff,88b2c554d5fffff,2026-01-01 09:00:00,2026-01-01 10:00:00,metro,study,1,1.0


,flow_id,movement_id
0,f_ab_bus_work_h08,m0
1,f_ab_bus_work_h08,m1
2,f_ac_metro_study_h09,m2


### Test 3.3 - Happy path integrado con reconstrucción desde trips_argument

Qué prueba: la función pública integra bien los helpers de reconstrucción exacta cuando no existe flow_to_trips. Verifica:

- fallback correcto a trips_argument,
- reconstrucción por llaves efectivas,
- presencia opcional de trip_id,
- summary correcto,
- no mutación ni de flows ni de trips.

In [27]:
trips = make_op13_test_tripdataset()
flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=False,
)

flows_before = snapshot_flowdataset_state(flows)
trips_before = snapshot_tripdataset_state(trips)

correspondence_df, report = get_trips_from_flows(
    flows,
    trips=trips,
    max_issues=20,
)

# Salida reconstruida
assert correspondence_df.columns.tolist() == ["flow_id", "movement_id", "trip_id"]
assert list(correspondence_df.itertuples(index=False, name=None)) == [
    ("f_ab_bus_work_h08", "m0", "t0"),
    ("f_ab_bus_work_h08", "m1", "t1"),
    ("f_ac_metro_study_h09", "m2", "t2"),
]

# Reporte
assert report.ok is True
assert report.parameters == {
    "max_issues": 20,
    "used_source": "trips_argument",
    "reconstruction_attempted": True,
    "n_flows_input": 2,
    "n_trips_input": 4,
}
assert report.summary == {
    "n_rows_out": 3,
    "n_unique_flows_out": 2,
    "n_unique_movements_out": 3,
    "n_unmatched_flows": 0,
    "n_unmatched_movements": 1,
}
assert_issue_codes(
    report.issues,
    ["GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE"],
)

# No side effects sobre flows
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
assert flows.flow_to_trips is flows_before["flow_to_trips"]
assert flows.aggregation_spec == flows_before["aggregation_spec"]
assert flows.metadata == flows_before["metadata"]
assert flows.provenance == flows_before["provenance"]

# No side effects sobre trips
pd.testing.assert_frame_equal(trips.data, trips_before["data"])
assert trips.schema is trips_before["schema"]
assert getattr(trips, "schema_effective", None) is trips_before["schema_effective"]
assert trips.metadata == trips_before["metadata"]
assert getattr(trips, "provenance", None) == trips_before["provenance"]

display(correspondence_df)
display(report.issues)
display(report.summary)


print("\nFlujos, viajes originales y viajes obtenidos:")
display(flows.flows)
display(trips.data)
display(correspondence_df)
# Ojo: aquí n_unmatched_movements == 1 porque la fixture base trae m3, que no cae en ningún flow del dataset de prueba.


,flow_id,movement_id,trip_id
0,f_ab_bus_work_h08,m0,t0
1,f_ab_bus_work_h08,m1,t1
2,f_ac_metro_study_h09,m2,t2


[Issue(level='warning', code='GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE', message='La correspondencia flujo-viajes quedó parcial: 1 movements y 0 flows quedaron sin correspondencia.', field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 2, 'n_trips_input': 4, 'used_source': 'trips_argument', 'reconstruction_attempted': True, 'source': 'trips_argument', 'join_key_columns': ['origin_h3_index', 'destination_h3_index', 'window_start_utc', 'window_end_utc', 'mode', 'purpose'], 'group_by': ['mode', 'purpose'], 'window_columns': ['window_start_utc', 'window_end_utc'], 'n_rows_out': 3, 'n_unmatched_movements': 1, 'n_unmatched_flows': 0, 'example_values': {'flow_id_sample_unmatched': [], 'movement_id_sample_unmatched': ['m3']}, 'reason': 'partial_coverage', 'action': 'return_partial_correspondence'})]

{'n_rows_out': 3,
 'n_unique_flows_out': 2,
 'n_unique_movements_out': 3,
 'n_unmatched_flows': 0,
 'n_unmatched_movements': 1}


Flujos, viajes originales y viajes obtenidos:


,flow_id,origin_h3_index,destination_h3_index,window_start_utc,window_end_utc,mode,purpose,flow_count,flow_value
0,f_ab_bus_work_h08,88b2c55417fffff,88b2c554cdfffff,2026-01-01 08:00:00,2026-01-01 09:00:00,bus,work,2,2.0
1,f_ac_metro_study_h09,88b2c55417fffff,88b2c554d5fffff,2026-01-01 09:00:00,2026-01-01 10:00:00,metro,study,1,1.0


,movement_id,trip_id,origin_h3_index,destination_h3_index,mode,purpose,origin_time_utc,destination_time_utc
0,m0,t0,88b2c55417fffff,88b2c554cdfffff,bus,work,2026-01-01T08:05:00Z,2026-01-01T08:25:00Z
1,m1,t1,88b2c55417fffff,88b2c554cdfffff,bus,work,2026-01-01T08:15:00Z,2026-01-01T08:35:00Z
2,m2,t2,88b2c55417fffff,88b2c554d5fffff,metro,study,2026-01-01T09:10:00Z,2026-01-01T09:35:00Z
3,m3,t3,88b2c55499fffff,88b2c55695fffff,bus,work,2026-01-01T08:20:00Z,2026-01-01T08:50:00Z


,flow_id,movement_id,trip_id
0,f_ab_bus_work_h08,m0,t0
1,f_ab_bus_work_h08,m1,t1
2,f_ac_metro_study_h09,m2,t2


### Test 3.4 - Degradado no fatal: flow_to_trips inválido y fallback a trips_argument

Qué prueba: el helper de resolución de fuente y la función pública integrados emiten warning, degradan correctamente al fallback y retornan una salida útil.

In [28]:
trips = make_op13_test_tripdataset()
flows = make_op13_test_flowdataset(
    include_flow_to_trips=True,
    include_extra_unmatched_flow=False,
)

# Se rompe el auxiliar directo para forzar la degradación.
flows.flow_to_trips = flows.flow_to_trips.loc[:, ["flow_id"]].copy()

flows_before = snapshot_flowdataset_state(flows)
trips_before = snapshot_tripdataset_state(trips)

correspondence_df, report = get_trips_from_flows(
    flows,
    trips=trips,
    max_issues=20,
)

codes = [issue.code for issue in report.issues]

# Debe haber caído a trips_argument
assert report.parameters["used_source"] == "trips_argument"
assert report.parameters["reconstruction_attempted"] is True
assert report.parameters["n_trips_input"] == 4

# Debe haberse emitido warning por fuente prioritaria inutilizable
assert "GET_TRIPS_FROM_FLOWS.SOURCE.PREFERRED_SOURCE_UNUSABLE" in codes

# Y además la salida debe ser válida y consistente
assert correspondence_df.columns.tolist() == ["flow_id", "movement_id", "trip_id"]
assert correspondence_df["movement_id"].tolist() == ["m0", "m1", "m2"]

# No mutación de inputs
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
pd.testing.assert_frame_equal(flows.flow_to_trips, flows_before["flow_to_trips"])
assert flows.metadata == flows_before["metadata"]

pd.testing.assert_frame_equal(trips.data, trips_before["data"])
assert trips.metadata == trips_before["metadata"]

display(correspondence_df)
display(report.issues)
display(report.parameters)

,flow_id,movement_id,trip_id
0,f_ab_bus_work_h08,m0,t0
1,f_ab_bus_work_h08,m1,t1
2,f_ac_metro_study_h09,m2,t2


[Issue(level='warning', code='GET_TRIPS_FROM_FLOWS.SOURCE.PREFERRED_SOURCE_UNUSABLE', message="La fuente prioritaria 'flow_to_trips' no fue usable; la operación continuó con 'trips_argument'.", field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 2, 'n_trips_input': None, 'used_source': 'trips_argument', 'reconstruction_attempted': False, 'preferred_source': 'flow_to_trips', 'checked_sources': ['flow_to_trips', 'trips_argument'], 'source_failures': {'flow_to_trips': {'reason': 'missing_required_columns', 'required_columns': ['flow_id', 'movement_id'], 'missing_columns': ['movement_id']}}, 'required_columns': ['flow_id', 'movement_id'], 'missing_columns': ['movement_id'], 'reason': 'preferred_source_unusable', 'action': 'use_fallback'}),
 Issue(level='warning', code='GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE', message='La correspondencia flujo-viajes quedó parcial: 1 movements y 0 flows quedaron sin correspondencia.', field=None, source_field=Non

{'max_issues': 20,
 'used_source': 'trips_argument',
 'reconstruction_attempted': True,
 'n_flows_input': 2,
 'n_trips_input': 4}

### Test 3.5 - Fatal de precondición: no existe ninguna fuente usable

Qué prueba: la función pública aborta correctamente cuando no hay de dónde construir la correspondencia. Este es el fatal más importante del contrato operativo de OP-13.

In [29]:
flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=False,
    source_trips=None,
)

flows_before = snapshot_flowdataset_state(flows)

raised = None
try:
    get_trips_from_flows(
        flows,
        trips=None,
        max_issues=20,
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, PylondrinaError)
assert raised.issue is not None
assert raised.issue.code == "GET_TRIPS_FROM_FLOWS.SOURCE.NO_USABLE_SOURCE"

# El input debe quedar completamente intacto
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
assert flows.flow_to_trips is flows_before["flow_to_trips"]
assert flows.aggregation_spec == flows_before["aggregation_spec"]
assert flows.metadata == flows_before["metadata"]
assert flows.provenance == flows_before["provenance"]

display(raised.issue)

Issue(level='error', code='GET_TRIPS_FROM_FLOWS.SOURCE.NO_USABLE_SOURCE', message='No existe ninguna fuente usable para construir la correspondencia flujo-viajes.', field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 2, 'n_trips_input': None, 'used_source': None, 'reconstruction_attempted': False, 'preferred_source': 'flow_to_trips', 'checked_sources': ['flow_to_trips', 'trips_argument', 'flows.source_trips'], 'source_failures': {'flow_to_trips': {'reason': 'missing', 'required_columns': ['flow_id', 'movement_id'], 'missing_columns': ['flow_id', 'movement_id']}, 'trips_argument': {'reason': 'missing', 'required_columns': ['movement_id', 'origin_h3_index', 'destination_h3_index'], 'missing_columns': ['movement_id', 'origin_h3_index', 'destination_h3_index']}, 'flows.source_trips': {'reason': 'missing', 'required_columns': ['movement_id', 'origin_h3_index', 'destination_h3_index'], 'missing_columns': ['movement_id', 'origin_h3_index', 'destination_h

### Test 3.6 - Resultado vacío retornable

Qué prueba: la operación fue interpretable, hubo fuente usable, pero ninguna correspondencia coincidió con los flows. Debe retornar tabla vacía, OperationReport, warning EMPTY_RESULT y cero side effects. Esto prueba una política clave del contrato.

In [30]:
trips = make_op13_test_tripdataset()

flows = make_op13_test_flowdataset(
    include_flow_to_trips=False,
    include_extra_unmatched_flow=False,
)

# Se reemplazan los flows por uno que no tenga matches con los trips.
flows.flows = pd.DataFrame(
    [
        {
            "flow_id": "f_nomatch",
            "origin_h3_index": flows.flows.loc[0, "destination_h3_index"],
            "destination_h3_index": flows.flows.loc[0, "origin_h3_index"],
            "window_start_utc": pd.Timestamp("2026-01-01 12:00:00"),
            "window_end_utc": pd.Timestamp("2026-01-01 13:00:00"),
            "mode": "walk",
            "purpose": "leisure",
            "flow_count": 1,
            "flow_value": 1.0,
        }
    ]
)

flows_before = snapshot_flowdataset_state(flows)
trips_before = snapshot_tripdataset_state(trips)

correspondence_df, report = get_trips_from_flows(
    flows,
    trips=trips,
    max_issues=20,
)

codes = [issue.code for issue in report.issues]

assert correspondence_df.empty is True
assert report.ok is True
assert "GET_TRIPS_FROM_FLOWS.OUTPUT.EMPTY_RESULT" in codes

assert report.summary == {
    "n_rows_out": 0,
    "n_unique_flows_out": 0,
    "n_unique_movements_out": 0,
    "n_unmatched_flows": 1,
    "n_unmatched_movements": 4,
}

# Sin side effects
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
assert flows.metadata == flows_before["metadata"]
pd.testing.assert_frame_equal(trips.data, trips_before["data"])
assert trips.metadata == trips_before["metadata"]

display(correspondence_df)
display(report.issues)
display(report.summary)

,flow_id,movement_id,trip_id


[Issue(level='warning', code='GET_TRIPS_FROM_FLOWS.OUTPUT.EMPTY_RESULT', message='La tabla de correspondencia flujo-viajes resultó vacía, aunque la operación fue interpretable.', field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 1, 'n_trips_input': 4, 'used_source': 'trips_argument', 'reconstruction_attempted': True, 'source': 'trips_argument', 'join_key_columns': ['origin_h3_index', 'destination_h3_index', 'window_start_utc', 'window_end_utc', 'mode', 'purpose'], 'group_by': ['mode', 'purpose'], 'window_columns': ['window_start_utc', 'window_end_utc'], 'n_rows_out': 0, 'n_unmatched_movements': 4, 'n_unmatched_flows': 1, 'example_values': {'flow_id_sample_unmatched': ['f_nomatch'], 'movement_id_sample_unmatched': ['m1', 'm2', 'm0', 'm3']}, 'reason': 'empty_result', 'action': 'return_empty_dataframe'})]

{'n_rows_out': 0,
 'n_unique_flows_out': 0,
 'n_unique_movements_out': 0,
 'n_unmatched_flows': 1,
 'n_unmatched_movements': 4}

### Test 3.7 - Fatal de preflight: falta flow_id en flows.flows

Qué prueba: aborta antes de resolver fuentes si el FlowDataset no cumple el contrato mínimo de entrada. Esto cubre otra precondición crítica del contrato.

In [31]:
flows = make_op13_test_flowdataset(
    include_flow_to_trips=True,
    include_extra_unmatched_flow=False,
)
flows.flows = flows.flows.drop(columns=["flow_id"]).copy()

flows_before = snapshot_flowdataset_state(flows)

raised = None
try:
    get_trips_from_flows(
        flows,
        max_issues=20,
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, PylondrinaError)
assert raised.issue is not None
assert raised.issue.code == "GET_TRIPS_FROM_FLOWS.DATA.MISSING_FLOW_ID"

# El input no cambia después del fallo
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
assert flows.metadata == flows_before["metadata"]

display(raised.issue)

Issue(level='error', code='GET_TRIPS_FROM_FLOWS.DATA.MISSING_FLOW_ID', message="El FlowDataset no contiene la columna requerida 'flow_id' en `flows.flows`.", field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 2, 'n_trips_input': None, 'used_source': None, 'reconstruction_attempted': False, 'field': 'flow_id', 'available_fields_sample': ['origin_h3_index', 'destination_h3_index', 'window_start_utc', 'window_end_utc', 'mode', 'purpose', 'flow_count', 'flow_value'], 'available_fields_total': 8, 'reason': 'missing_required_column', 'action': 'abort'})

### Test 3.8 - Truncamiento explícito del reporte

Qué prueba: si hay más hallazgos que max_issues, el reporte se trunca, aparece el issue final de truncamiento y summary["limits"] queda consistente. En OP-13 el truncamiento es importante porque el reporte debe seguir pequeño y estable.


In [35]:
trips = make_op13_test_tripdataset()
flows = make_op13_test_flowdataset(
    include_flow_to_trips=True,
    include_extra_unmatched_flow=True,
)

# Se rompe el flow_to_trips para forzar fallback + warning
flows.flow_to_trips = flows.flow_to_trips.loc[:, ["flow_id"]].copy()

correspondence_df, report = get_trips_from_flows(
    flows,
    trips=trips,
    max_issues=1,
)

codes = [issue.code for issue in report.issues]
# Debe haber truncamiento explícito
assert "GET_TRIPS_FROM_FLOWS.REPORT.ISSUES_TRUNCATED" in codes
assert "limits" in report.summary

limits = report.summary["limits"]
assert limits["max_issues"] == 1
assert limits["issues_truncated"] is True
assert limits["n_issues_detected_total"] >= 2
assert limits["n_issues_emitted"] == 1

# Aunque el reporte se haya truncado, la salida tabular sigue siendo coherente
assert correspondence_df.columns.tolist() == ["flow_id", "movement_id", "trip_id"]
assert report.parameters["used_source"] == "trips_argument"

display(correspondence_df)
display(report.issues)
display(report.summary)

,flow_id,movement_id,trip_id
0,f_ab_bus_work_h08,m0,t0
1,f_ab_bus_work_h08,m1,t1
2,f_ac_metro_study_h09,m2,t2


[Issue(level='warning', code='GET_TRIPS_FROM_FLOWS.REPORT.ISSUES_TRUNCATED', message='La lista de issues de get_trips_from_flows fue truncada por max_issues=1.', field=None, source_field=None, row_count=None, details={'max_issues': 1, 'n_issues_emitted': 1, 'n_issues_detected_total': 2, 'issues_truncated': True, 'action': 'truncate_issues'})]

{'n_rows_out': 3,
 'n_unique_flows_out': 2,
 'n_unique_movements_out': 3,
 'n_unmatched_flows': 1,
 'n_unmatched_movements': 1,
 'limits': {'max_issues': 1,
  'issues_truncated': True,
  'n_issues_emitted': 1,
  'n_issues_detected_total': 2}}